# 🤖 ML Salary Predictor
### NTUC LearningHub — Mini Project 1
**Skills:** scikit-learn · Feature Engineering · Cross-Validation · Model Comparison · Feature Importance  
**Algorithm family:** Regression (Linear, Ridge, Random Forest, Gradient Boosting)
---


## 1️⃣ Imports & Setup

In [ ]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score

sns.set_theme(style="whitegrid"); Path("ml_output").mkdir(exist_ok=True)
print("✅ Imports ready")

## 2️⃣ Load Data

In [ ]:
try:
    df = pd.read_csv("clean_data.csv")
    print(f"✅ Loaded clean_data.csv — {df.shape}")
except FileNotFoundError:
    print("ℹ  clean_data.csv not found — generating synthetic data for demo")
    np.random.seed(42); n=200
    edu_mult  = {"Diploma":0.8,"Bachelor's":1.0,"Master's":1.2,"PhD":1.5}
    dept_mult = {"IT":1.1,"Finance":1.15,"HR":0.9,"Data Science":1.2,"Marketing":0.95}
    df = pd.DataFrame({
        "Department"      : np.random.choice(list(dept_mult), n),
        "Gender"          : np.random.choice(["Male","Female"], n),
        "Education Level" : np.random.choice(list(edu_mult), n, p=[0.25,0.35,0.30,0.10]),
        "Seniority Band"  : np.random.choice(["Junior","Mid-Level","Senior","Expert"], n, p=[0.25,0.35,0.25,0.15]),
        "Years Experience": np.random.randint(1,30,n),
        "Age"             : np.random.randint(22,58,n),
        "Performance Score": np.random.uniform(2.5,5.0,n).round(1),
    })
    df["Salary"] = (3000 + df["Years Experience"]*320 + df["Age"]*50
                    + df["Performance Score"]*400
                    + df["Education Level"].map(edu_mult)*1500
                    + df["Department"].map(dept_mult)*1000
                    + np.random.normal(0,800,n)).round(0)
    print(f"✅ Synthetic dataset: {df.shape}")
df.head()

## 3️⃣ Feature Engineering & Encoding

In [ ]:
features = ["Department","Gender","Education Level","Seniority Band",
            "Years Experience","Age","Performance Score"]

df_ml = df[features + ["Salary"]].dropna().copy()

le = LabelEncoder()
for col in ["Department","Gender","Education Level","Seniority Band"]:
    df_ml[col] = le.fit_transform(df_ml[col].astype(str))

X = df_ml[features].values
y = df_ml["Salary"].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")

## 4️⃣ Model Training & Comparison

In [ ]:
models = {
    "Linear Regression" : LinearRegression(),
    "Ridge Regression"  : Ridge(alpha=10),
    "Random Forest"     : RandomForestRegressor(n_estimators=100, random_state=42),
    "Gradient Boosting" : GradientBoostingRegressor(n_estimators=100, random_state=42),
}

results = []
for name, model in models.items():
    pipe = Pipeline([("scaler", StandardScaler()), ("model", model)])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    mae  = mean_absolute_error(y_test, y_pred)
    r2   = r2_score(y_test, y_pred)
    cv   = cross_val_score(pipe, X_train, y_train, cv=5, scoring="r2").mean()
    results.append({"Model":name, "MAE":round(mae,0), "R²":round(r2,4), "CV R²":round(cv,4)})
    print(f"{name:<25} MAE={mae:,.0f}  R²={r2:.4f}  CV R²={cv:.4f}")

results_df = pd.DataFrame(results)
display(results_df)

## 5️⃣ Feature Importance (Random Forest)

In [ ]:
rf_pipe = Pipeline([("scaler",StandardScaler()),
                    ("model",RandomForestRegressor(n_estimators=100, random_state=42))])
rf_pipe.fit(X_train, y_train)

fi = pd.DataFrame({"Feature":features,
                   "Importance":rf_pipe.named_steps["model"].feature_importances_})
fi = fi.sort_values("Importance")

fig, ax = plt.subplots(figsize=(8,5))
ax.barh(fi["Feature"], fi["Importance"], color=sns.color_palette("Set2",len(fi)), edgecolor="white")
ax.set_title("Feature Importance — Random Forest Salary Predictor", fontweight="bold")
ax.set_xlabel("Importance Score")
plt.tight_layout(); plt.show()

## 6️⃣ Actual vs Predicted

In [ ]:
best_pipe = Pipeline([("scaler",StandardScaler()),
                      ("model",GradientBoostingRegressor(n_estimators=100, random_state=42))])
best_pipe.fit(X_train, y_train)
y_pred = best_pipe.predict(X_test)

fig, ax = plt.subplots(figsize=(8,6))
ax.scatter(y_test, y_pred, alpha=0.7, color="#4e9af1", edgecolors="white", s=60)
lim = [min(y_test.min(), y_pred.min())-500, max(y_test.max(), y_pred.max())+500]
ax.plot(lim, lim, "r--", lw=2, label="Perfect Prediction")
ax.set_xlabel("Actual Salary"); ax.set_ylabel("Predicted Salary")
ax.set_title("Actual vs Predicted Salary — Gradient Boosting", fontweight="bold")
ax.legend(); plt.tight_layout(); plt.show()
print(f"R² = {r2_score(y_test,y_pred):.4f}  |  MAE = SGD {mean_absolute_error(y_test,y_pred):,.0f}")